# Session A: bench, activations, fit, bridge, ablations
Runtime: GPU (A100, high RAM). Add `HF_TOKEN` under Colab secrets, then Run all.

In [ ]:
!pip -q install "transformers>=5.15,<6" sacrebleu sentencepiece safetensors pyyaml curatorkit

In [ ]:
import os, sys, subprocess
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("Colab secret HF_TOKEN not found; expecting HF_TOKEN in the environment.", e)
from huggingface_hub import whoami, snapshot_download
USER = whoami()["name"]
os.environ["XLRB_HF_USER"] = USER
os.environ["XLRB_WORK"] = "/content/xlrb_work"
code = snapshot_download(f"{USER}/xlrb-ops", repo_type="dataset", allow_patterns=["code/*"], local_dir="/content/xlrb_code")
os.environ["PYTHONPATH"] = "/content/xlrb_code/code/src"
os.environ["XLRB_CODE_SHA"] = open("/content/xlrb_code/code/CODE_SHA").read().strip()
print("user:", USER, "| code:", os.environ["XLRB_CODE_SHA"])
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Idempotent: if the runtime disconnects, re-run setup and this cell; finished stages are skipped.
!cd /content && python -u -m xlrb.cli run_a 2>&1 | tee -a /content/session_a.log
from huggingface_hub import HfApi
import os, time
HfApi().upload_file(path_or_fileobj="/content/session_a.log", path_in_repo=f"logs/session_a_{time.strftime('%Y%m%dT%H%M%S')}.log",
                    repo_id=f"{os.environ['XLRB_HF_USER']}/xlrb-ops", repo_type="dataset")